# Colab com o estudo dos dados do BCB para o projeto de API do 3° Semestre para a DM

O objetivo desse colab é baixar os dados vindos do gov, filtrar os mesmos e analisar o que é preciso para inferir regiões com alta capacidade de expansão.






In [ ]:
# Baixa o arquivo ZIP do SCR do ano anterior diretamente do Banco Central.
# O ano é calculado dinamicamente para não precisar atualizar o código manualmente.
# O download é feito em chunks de 8KB para não sobrecarregar a memória.

import datetime
import requests

def baixar_zip():
    ano_anterior = datetime.datetime.now().year - 1
    url = f"https://www.bcb.gov.br/pda/desig/scrdata_{ano_anterior}.zip"
    filename = f"scrdata_{ano_anterior}.zip"

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    return filename

filename = baixar_zip()

In [ ]:
# Lê todos os CSVs dentro do zip do BCB, filtrando apenas as colunas necessárias.
# Converte data_base para datetime e as colunas numéricas de string com vírgula para float.
# Concatena todos os meses em um único DataFrame e libera a lista intermediária da memória.

import pandas as pd
import zipfile

colunas = [
    "data_base", "uf", "numero_de_operacoes",
    "carteira_ativa", "carteira_inadimplencia",
    "ativo_problematico", "porte", "cliente",
]

colunas_numericas = [
    "carteira_ativa",
    "carteira_inadimplencia",
    "ativo_problematico",
]

dfs = []

with zipfile.ZipFile(filename) as zf:
    for nome in zf.namelist():
        with zf.open(nome) as file:
            df = pd.read_csv(file, sep=";", usecols=colunas)
            df["numero_de_operacoes"] = df["numero_de_operacoes"].replace(-1, pd.NA)

            df = df.assign(
                data_base=pd.to_datetime(df["data_base"]),
                **{
                    col: pd.to_numeric(
                        df[col].astype(str).str.replace(",", ".", regex=False),
                        errors="coerce",
                    )
                    for col in colunas_numericas
                }
            )

            df = df[df["porte"] != "Indisponível"]

            # vai agrupar pelo by= e depois somar os valores numéricos, o as_index=false faz com que os valores do by= não se tornem indices
            df_agrupado = df.groupby(
                by=["data_base", "uf", "porte", "cliente"],
                as_index=False
            )[["numero_de_operacoes", "carteira_ativa", "carteira_inadimplencia", "ativo_problematico"]].sum()
            dfs.append(df_agrupado)

dt = pd.concat(dfs, ignore_index=True)
del dfs


In [ ]:
# Calcula o total da carteira ativa nacional e exibe de forma legível.
# formatar_valor() abrevia valores grandes em K, M, B ou T
# para facilitar a leitura de números na casa dos bilhões e trilhões.

def formatar_valor(valor):
    if valor >= 1_000_000_000_000:
        return f"{valor / 1_000_000_000_000:.2f}T"
    elif valor >= 1_000_000_000:
        return f"{valor / 1_000_000_000:.2f}B"
    elif valor >= 1_000_000:
        return f"{valor / 1_000_000:.2f}M"
    elif valor >= 1_000:
        return f"{valor / 1_000:.2f}K"
    return f"{valor:.2f}"

carteira_ativa_totalNacional = dt['carteira_ativa'].sum()
print(formatar_valor(carteira_ativa_totalNacional))

81.56T


In [ ]:
# Calcula a variação percentual da carteira ativa entre o primeiro e o último mês do período.
# Agrupa o total da carteira ativa por mês, pega o valor inicial (iloc[0]) e o final (iloc[-1]),
# e aplica a fórmula: (final - inicial) / inicial * 100.

carteira_por_mes = dt.groupby('data_base')['carteira_ativa'].sum().sort_index()

valor_inicial = carteira_por_mes.iloc[0]
valor_final = carteira_por_mes.iloc[-1]

variacao = (valor_final - valor_inicial) / valor_inicial * 100

print(f"Variação da carteira ativa: {round(variacao, 2)}%")

Variação da carteira ativa: 11.57%


In [ ]:
# Calcula a taxa de inadimplência nacional pelo total do período.
# Fórmula: (carteira_inadimplencia / carteira_ativa) * 100
# O resultado representa o percentual da carteira ativa com parcelas vencidas há mais de 90 dias.

carteira_ativa_totalNacional = dt['carteira_ativa'].sum()
carteira_inadimplencia_totalNacional = dt['carteira_inadimplencia'].sum()
taxa_inadimplecia_nacional = round(carteira_inadimplencia_totalNacional / carteira_ativa_totalNacional * 100, 2)

print(taxa_inadimplecia_nacional)

3.81


In [ ]:
# Calcula o total de operações de crédito no período, ignorando os valores NaN
# que foram substituídos anteriormente (numero_de_operacoes = -1).

numOperacoesTotal = dt['numero_de_operacoes'].sum(skipna=True)
print(formatar_valor(numOperacoesTotal))

9.82B


In [ ]:
# Estado com maior carteira ativa
# Agrupa os dados por UF, soma as carteiras ativas e identifica o estado em destaque

carteira_por_estado = dt.groupby('uf')['carteira_ativa'].sum().sort_index()

estado_destaque = carteira_por_estado.idxmax()
print(f"Estado em destaque: {estado_destaque}")

Estado em destaque: SP


In [ ]:
# Estado com maior crescimento de carteira ativa (a.a.)
# Agrupa os dados por UF e mês, calcula a variação percentual entre o primeiro e o último mês
# do período (12 meses) para cada estado e identifica o estado com maior crescimento anual.

carteira_por_estado_mes = dt.groupby(['uf', 'data_base'])['carteira_ativa'].sum().sort_index()

valor_inicial = carteira_por_estado_mes.groupby('uf').first()
valor_final = carteira_por_estado_mes.groupby('uf').last()

variacao_por_estado = (valor_final - valor_inicial) / valor_inicial * 100

estado_maior_crescimento = variacao_por_estado.idxmax()
valor_maior_crescimento = variacao_por_estado.max()

print(f"Estado com maior crescimento: {estado_maior_crescimento} ({round(valor_maior_crescimento, 2)}%)")

Estado com maior crescimento: TO (17.99%)


In [ ]:
carteira_por_estado_mes = dt.groupby(['uf', 'data_base'])['carteira_ativa'].sum().reset_index()
carteira_por_estado_mes = carteira_por_estado_mes.sort_values(['uf', 'data_base'])

def crescimento_mensal(grupo):
    if len(grupo) < 2:
        return None
    atual = grupo['carteira_ativa'].iloc[-1]
    anterior = grupo['carteira_ativa'].iloc[-2]
    if anterior == 0:
        return None
    return ((atual - anterior) / anterior) * 100

variacao = (
    carteira_por_estado_mes
    .groupby('uf')
    .apply(crescimento_mensal)
    .dropna()
)

estado_maior_crescimento = variacao.idxmax()
valor_maior_crescimento = variacao.max()

print(f"Estado com maior crescimento: {estado_maior_crescimento} ({round(valor_maior_crescimento, 2)}%)")

Estado com maior crescimento: DF (6.35%)


/tmp/ipykernel_3942/4056888443.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(crescimento_mensal)


In [ ]:
# Exploração dos dados — não faz parte da análise final.

# pequenas informações sobre cada uma das colunas, como seus tipos de dados, quantos conteudos não nulos possuem entre outros.

dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3888 entries, 0 to 3887
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   data_base               3888 non-null   datetime64[ns]
 1   uf                      3888 non-null   object        
 2   porte                   3888 non-null   object        
 3   cliente                 3888 non-null   object        
 4   numero_de_operacoes     3888 non-null   object        
 5   carteira_ativa          3888 non-null   float64       
 6   carteira_inadimplencia  3888 non-null   float64       
 7   ativo_problematico      3888 non-null   float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 243.1+ KB


In [ ]:
# Exploração dos dados — não faz parte da análise final.

# imprime quantas linhas tem para cada tipo de porte dos clientes PF.

print(dt[dt["cliente"] == "PF"]["porte"].value_counts())


porte
Acima de 20 salários mínimos        324
Até 1 salário mínimo                324
Mais de 1 a 2 salários mínimos      324
Mais de 10 a 20 salários mínimos    324
Mais de 2 a 3 salários mínimos      324
Mais de 3 a 5 salários mínimos      324
Mais de 5 a 10 salários mínimos     324
Sem rendimento                      324
Name: count, dtype: int64


In [ ]:
# Exploração dos dados — não faz parte da análise final.

# imprime quantas linhas tem para cada tipo de porte dos clientes PJ

print(dt[dt["cliente"] == "PJ"]["porte"].value_counts())

porte
Grande     324
Micro      324
Médio      324
Pequeno    324
Name: count, dtype: int64


# Tipos de porte PF
- Acima de 20 salários mínimos        
- Até 1 salário mínimo                
- Mais de 1 a 2 salários mínimos      
- Mais de 10 a 20 salários mínimos    
- Mais de 2 a 3 salários mínimos      
- Mais de 3 a 5 salários mínimos      
- Mais de 5 a 10 salários mínimos     
- Sem rendimento  

# Tipos de porte PJ

- Grande     
- Micro      
- Médio     
- Pequeno    

In [ ]:
# Exploração dos dados — não faz parte da análise final.
dt[dt["cliente"] == "PF"][["uf"]].value_counts()

,count
uf,
AC,96
AL,96
AM,96
AP,96
BA,96
CE,96
DF,96
ES,96
GO,96


In [ ]:
# Exploração dos dados — não faz parte da análise final.
dt[(dt["cliente"] == "PF") & (dt["uf"] == "SP")][["porte","uf"]].value_counts()

,,count
porte,uf,
Acima de 20 salários mínimos,SP,12
Até 1 salário mínimo,SP,12
Mais de 1 a 2 salários mínimos,SP,12
Mais de 10 a 20 salários mínimos,SP,12
Mais de 2 a 3 salários mínimos,SP,12
Mais de 3 a 5 salários mínimos,SP,12
Mais de 5 a 10 salários mínimos,SP,12
Sem rendimento,SP,12


In [ ]:
# Exploração dos dados — não faz parte da análise final.
dt["data_base"].value_counts().sort_index()

,count
data_base,
2025-01-31,324
2025-02-28,324
2025-03-31,324
2025-04-30,324
2025-05-31,324
2025-06-30,324
2025-07-31,324
2025-08-31,324
2025-09-30,324


In [ ]:
# Exploração dos dados — não faz parte da análise final.

import pandas as pd
import numpy as np

# Helpers

populacao_estados = {
    "RO": 1751950,
    "AC": 884372,
    "AM": 4321616,
    "RR": 738772,
    "PA": 8711196,
    "AP": 806517,
    "TO": 1586859,

    "MA": 7018211,
    "PI": 3384547,
    "CE": 9268836,
    "RN": 3455236,
    "PB": 4164468,
    "PE": 9562007,
    "AL": 3220848,
    "SE": 2299425,
    "BA": 14870907,

    "MG": 21393441,
    "ES": 4126854,
    "RJ": 17223547,
    "SP": 46081801,

    "PR": 11890517,
    "SC": 8187029,
    "RS": 11233263,

    "MS": 2924631,
    "MT": 3893659,
    "GO": 7423629,
    "DF": 2996899
}

# ============================================================
# ANÁLISE DE EXPANSÃO SUSTENTÁVEL DE CRÉDITO — POR ESTADO
# ============================================================
# Pré-requisito: `dt` já carregado e limpo conforme os scripts anteriores.
# Resultado: DataFrame `df_estados` com 27 linhas (uma por UF), indicadores
# de crédito e um score final de 0–100 para priorizar expansão.
#
# Lógica do score (inspirada na tabela de referência):
#   Demanda alta       → sobe o score  (+)
#   Inadimplência alta → derruba score (-)
#   Endividamento alto → derruba score (-)
#
# Pesos:
#   Demanda (número de operações)   → 35%
#   Crescimento da carteira         → 15%
#   Taxa de inadimplência           → 30%  (penaliza)
#   Taxa de ativo problemático      → 20%  (penaliza — proxy de endividamento)
# ============================================================


# ------------------------------------------------------------------
# 1. VOLUME TOTAL DE CRÉDITO POR ESTADO
# ------------------------------------------------------------------
carteira_por_estado = (
    dt.groupby("uf")["carteira_ativa"]
    .sum()
    .rename("carteira_ativa_total")
)

# ------------------------------------------------------------------
# 2. CRESCIMENTO DA CARTEIRA (dois períodos mais recentes)
# ------------------------------------------------------------------
datas_disponiveis = dt["data_base"].sort_values().unique()

if len(datas_disponiveis) >= 2:
    data_atual    = datas_disponiveis[-1]
    data_anterior = datas_disponiveis[-7]

    carteira_atual = (
        dt[dt["data_base"] == data_atual]
        .groupby("uf")["carteira_ativa"]
        .sum()
        .rename("carteira_periodo_atual")
    )
    carteira_anterior = (
        dt[dt["data_base"] == data_anterior]
        .groupby("uf")["carteira_ativa"]
        .sum()
        .rename("carteira_periodo_anterior")
    )
    carteira_anterior_safe = carteira_anterior.replace(0, np.nan)
    crescimento = (
        ((carteira_atual - carteira_anterior) / carteira_anterior_safe * 100)
        .rename("crescimento_carteira_pct")
    )
    crescimento_ajustado = (
    (
        crescimento * np.log1p(carteira_por_estado)
    )
    .pipe(lambda s: s.clip(
        lower=s.quantile(0.05),
        upper=s.quantile(0.95)
    ))
    .rename("crescimento_ajustado")
)
else:
    crescimento = pd.Series(dtype=float, name="crescimento_carteira_pct")

# ------------------------------------------------------------------
# 3. TAXA DE INADIMPLÊNCIA
#    Soma dos valores → depois divide (evita média de médias)
# ------------------------------------------------------------------
inadimplencia_total = (
    dt.groupby("uf")["carteira_inadimplencia"]
    .sum()
    .rename("inadimplencia_total")
)

taxa_inadimplencia = (
    (inadimplencia_total / carteira_por_estado * 100)
    .rename("taxa_inadimplencia_pct")
)

# ------------------------------------------------------------------
# 4. ATIVO PROBLEMÁTICO (proxy de endividamento/risco latente)
# ------------------------------------------------------------------
ativo_problematico_total = (
    dt.groupby("uf")["ativo_problematico"]
    .sum()
    .rename("ativo_problematico_total")
)

taxa_ativo_problematico = (
    (ativo_problematico_total / carteira_por_estado * 100)
    .rename("taxa_ativo_problematico_pct")
)

# ------------------------------------------------------------------
# 5. NÚMERO DE OPERAÇÕES (proxy de demanda)
# ------------------------------------------------------------------
operacoes_total = (
    dt.groupby("uf")["numero_de_operacoes"]
    .sum(min_count=1)
    .rename("numero_de_operacoes_total")
)

# ------------------------------------------------------------------
# 6. MONTANDO O DATAFRAME BASE
# ------------------------------------------------------------------
df_estados = (
    pd.concat(
        [
            carteira_por_estado,
            crescimento,
            crescimento_ajustado,
            inadimplencia_total,
            taxa_inadimplencia,
            ativo_problematico_total,
            taxa_ativo_problematico,
            operacoes_total,
        ],
        axis=1,
    )
    .reset_index()
    .rename(columns={"uf": "estado"})
    .reset_index(drop=True)
)

# Obs: dados de população e operações por habitante - mede a penetração do credito

df_estados["populacao"] = df_estados["estado"].map(populacao_estados)

df_estados["operacoes_por_habitante"] = (
    df_estados["numero_de_operacoes_total"] / df_estados["populacao"]
)

df_estados["operacoes_por_habitante"] = pd.to_numeric(
    df_estados["operacoes_por_habitante"],
    errors="coerce"
)

df_estados["operacoes_por_habitante"] = df_estados["operacoes_por_habitante"].fillna(0)

df_estados["demanda_ajustada"] = np.log1p(df_estados["operacoes_por_habitante"])

# ------------------------------------------------------------------
# 7. SISTEMA DE SCORE DE EXPANSÃO (0–100)
# ------------------------------------------------------------------
# Estratégia: Min-Max Normalization
#
#   Cada indicador é normalizado para 0–100 dentro do grupo de estados,
#   permitindo comparação justa independente da escala original.
#
#   Indicadores POSITIVOS (mais = melhor):
#     → normalização direta: 0 = pior estado, 100 = melhor estado
#
#   Indicadores NEGATIVOS (mais = pior):
#     → normalização invertida: 0 = pior estado, 100 = melhor estado
#
#   Score final = soma ponderada dos componentes normalizados.
#
# Pesos definidos:
PESO_DEMANDA        = 0.35   # número de operações          → (+) quanto mais demanda, melhor
PESO_CRESCIMENTO    = 0.15   # crescimento da carteira       → (+) momentum positivo
PESO_INADIMPLENCIA  = 0.30   # taxa de inadimplência         → (-) risco de calote
PESO_PROBLEMATICO   = 0.20   # taxa de ativo problemático    → (-) endividamento/risco latente
# Total = 1.00

def normalizar(serie, inverter=False):
    """
    Normaliza uma Series para o intervalo 0–100 via Min-Max.
    inverter=True  → score alto = valor baixo (penaliza indicadores ruins).
    inverter=False → score alto = valor alto  (premia indicadores bons).
    NaNs são preenchidos com a mediana antes de normalizar.
    """
    s = serie.fillna(serie.median())
    minimo, maximo = s.min(), s.max()
    if maximo == minimo:
        # Todos os estados com mesmo valor → score neutro 50
        return pd.Series(50.0, index=serie.index)
    normalizado = (s - minimo) / (maximo - minimo) * 100
    return (100 - normalizado) if inverter else normalizado

# Componentes normalizados individualmente
score_demanda       = normalizar(df_estados["demanda_ajustada"],   inverter=True)
score_crescimento   = normalizar(df_estados["crescimento_ajustado"],    inverter=False)
score_inadimplencia = normalizar(df_estados["taxa_inadimplencia_pct"],      inverter=True)
score_problematico  = normalizar(df_estados["taxa_ativo_problematico_pct"], inverter=True)

# Score final ponderado
df_estados["score_expansao"] = (
      score_demanda       * PESO_DEMANDA
    + score_crescimento   * PESO_CRESCIMENTO
    + score_inadimplencia * PESO_INADIMPLENCIA
    + score_problematico  * PESO_PROBLEMATICO
).round(1)

# ------------------------------------------------------------------
# 8. CLASSIFICAÇÃO QUALITATIVA
#    Traduz o score numérico em categorias para leitura rápida.
#
#    🟢 Ótimo   → score ≥ 75  — expansão recomendada
#    🟡 Bom     → score 60–74 — expansão com atenção
#    🟠 Regular → score 45–59 — expansão cautelosa
#    🔴 Evitar  → score < 45  — risco alto, não recomendado
# ------------------------------------------------------------------
def classificar(score):
    if score >= 75:
        return "🟢 Ótimo"
    elif score >= 60:
        return "🟡 Bom"
    elif score >= 45:
        return "🟠 Regular"
    else:
        return "🔴 Evitar"

df_estados["classificacao"] = df_estados["score_expansao"].apply(classificar)

# Ordena do melhor para o pior candidato à expansão
df_estados = df_estados.sort_values("score_expansao", ascending=False).reset_index(drop=True)

# ------------------------------------------------------------------
# 9. RESULTADO FINAL
# ------------------------------------------------------------------
print("=" * 65)
print("RANKING DE EXPANSÃO POR ESTADO")
print("=" * 65)
print(
    df_estados[[
        "estado",
        "score_expansao",
        "classificacao",
        "taxa_inadimplencia_pct",
        "taxa_ativo_problematico_pct",
        "numero_de_operacoes_total",
        "crescimento_carteira_pct",
        "crescimento_ajustado"
    ]].to_string(index=False)
)
print()
print(f"Total de estados analisados: {len(df_estados)}")
print()
print("Distribuição das classificações:")
print(df_estados["classificacao"].value_counts().to_string())

RANKING DE EXPANSÃO POR ESTADO
estado  score_expansao classificacao  taxa_inadimplencia_pct  taxa_ativo_problematico_pct numero_de_operacoes_total  crescimento_carteira_pct  crescimento_ajustado
    AM            70.3         🟡 Bom                4.595932                     7.455995                 136488182                  8.594855            235.997681
    PI            66.6         🟡 Bom                3.967245                     6.636617                 117639188                  4.066822            111.087794
    DF            65.6         🟡 Bom                3.057686                     5.680531                 167602921                 19.062853            358.666435
    SC            65.3         🟡 Bom                2.999236                     5.597203                 399484136                  6.892970            200.756078
    CE            64.4         🟡 Bom                4.446364                     7.670630                 345504637                 10.307776        

In [ ]:
# Exploração dos dados — não faz parte da análise final.

# =============================================================================
# ANÁLISE DE EXPANSÃO SUSTENTÁVEL DE CRÉDITO — POR ESTADO  (v2)
# =============================================================================
# Autor       : Guilherme
# Fonte       : Banco Central do Brasil
# Pré-requisito: `dt` carregado, limpo e tipado.
#
# Mudanças principais em relação à v1:
#   • Memória         — aggregações feitas em único groupby/agg (elimina
#                       DataFrames intermediários); uso explícito de dtypes
#                       enxutos (float32) pós-concat.
#   • Pilar Demanda   → renomeado para "Penetração/Disponibilidade":
#                       usa operacoes_por_habitante (já existia) + NOVO
#                       saldo_per_capita (carteira_ativa / população) invertido.
#   • Pilar Risco     → Índice de Estresse de Crédito (IEC):
#                       combina taxa_inadimplencia + ativo_problematico e aplica
#                       penalidade extra a estados acima da média nacional.
#   • Pilar Crescimento → valida se o crescimento do saldo é acompanhado pelo
#                       crescimento de operações (evita concentração de risco).
#   • Filtro PF / PJ  — score calculado em separado por segmento (client_type).
#   • Janela de crescimento fixada em [-1] vs [-7] (≈ 6 meses).
#   • Nomenclatura e comentários alinhados à nova semântica.
# =============================================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# CONFIGURAÇÕES GLOBAIS
# -----------------------------------------------------------------------------
CLIENTE_FILTRO: str | None = None   # None = todos | "PF" | "PJ"

PESO_PENETRACAO   = 0.35   # penetração / disponibilidade (+)
PESO_CRESCIMENTO  = 0.15   # crescimento validado          (+)
PESO_ESTRESSE     = 0.30   # índice de estresse de crédito (-)
PESO_ENDIVIDAMENTO = 0.20  # saldo per capita              (-)
# Soma = 1.00

PENALIDADE_ACIMA_MEDIA = 10.0  # pontos descontados do score se IEC > média BR

# -----------------------------------------------------------------------------
# POPULAÇÃO POR UF (fonte: IBGE — conforme dicionário original do projeto)
# -----------------------------------------------------------------------------
POPULACAO_ESTADOS: dict[str, int] = {
    "RO": 1751950,  "AC": 884372,   "AM": 4321616,  "RR": 738772,
    "PA": 8711196,  "AP": 806517,   "TO": 1586859,
    "MA": 7018211,  "PI": 3384547,  "CE": 9268836,  "RN": 3455236,
    "PB": 4164468,  "PE": 9562007,  "AL": 3220848,  "SE": 2299425,
    "BA": 14870907,
    "MG": 21393441, "ES": 4126854,  "RJ": 17223547, "SP": 46081801,
    "PR": 11890517, "SC": 8187029,  "RS": 11233263,
    "MS": 2924631,  "MT": 3893659,  "GO": 7423629,  "DF": 2996899,
}

# =============================================================================
# ETAPA 0 — FILTRO OPCIONAL DE SEGMENTO (PF / PJ)
# =============================================================================
# Filtragem antecipada reduz o volume de dados antes de qualquer aggregação,
# poupando memória nas etapas seguintes.
# Ajuste a coluna "cliente" para o nome real no seu dataset.

if CLIENTE_FILTRO is not None:
    _col_cliente = "cliente"          # ← ajuste se necessário
    if _col_cliente not in dt.columns:
        raise KeyError(
            f"Coluna '{_col_cliente}' não encontrada. "
            "Ajuste `_col_cliente` ou defina CLIENTE_FILTRO = None."
        )
    dt = dt[dt[_col_cliente] == CLIENTE_FILTRO].copy()
    print(f"[INFO] Segmento filtrado: {CLIENTE_FILTRO}")

# =============================================================================
# ETAPA 1 — AGGREGAÇÃO ÚNICA (memória otimizada)
# =============================================================================
# Um único groupby/agg por UF evita criar múltiplos DataFrames intermediários.
# O resultado é um DataFrame com todas as colunas numéricas necessárias.

_agg_total = (
    dt.groupby("uf", sort=False)
    .agg(
        carteira_ativa_total   =("carteira_ativa",         "sum"),
        inadimplencia_total    =("carteira_inadimplencia", "sum"),
        ativo_problematico_total=("ativo_problematico",    "sum"),
        numero_de_operacoes_total=("numero_de_operacoes",  "sum"),
    )
    .reset_index()
    .rename(columns={"uf": "estado"})
)

# Downcasting para float32 economiza ~50 % de memória em colunas numéricas.
_colunas_float = [
    "carteira_ativa_total",
    "inadimplencia_total",
    "ativo_problematico_total",
    "numero_de_operacoes_total",
]
_agg_total[_colunas_float] = _agg_total[_colunas_float].astype("float32")

# =============================================================================
# ETAPA 2 — CRESCIMENTO DA CARTEIRA (janela de ~6 meses)
# =============================================================================
# Índice [-1] = período mais recente; [-7] = seis meses atrás.
# Valida também o crescimento de operações para detectar concentração.

datas_disponiveis = np.sort(dt["data_base"].unique())

if len(datas_disponiveis) >= 7:
    data_atual    = datas_disponiveis[-1]
    data_anterior = datas_disponiveis[-7]

    # Aggregação dos dois cortes de data em um único passo
    _snapshot = (
        dt[dt["data_base"].isin([data_atual, data_anterior])]
        .groupby(["uf", "data_base"], sort=False)
        [["carteira_ativa", "numero_de_operacoes"]]
        .sum()
        .unstack("data_base")          # → MultiIndex de colunas
    )

    # Acesso seguro às colunas independentemente da ordem
    _carteira_ant  = _snapshot[("carteira_ativa",         data_anterior)]
    _carteira_atu  = _snapshot[("carteira_ativa",         data_atual)]
    _operacoes_ant = _snapshot[("numero_de_operacoes",    data_anterior)]
    _operacoes_atu = _snapshot[("numero_de_operacoes",    data_atual)]

    # Crescimento nominal do saldo (%)
    _cresc_saldo = (
        (_carteira_atu - _carteira_ant)
        / _carteira_ant.replace(0, np.nan)
        * 100
    )

    # Crescimento de operações (%)
    _cresc_ops = (
        (_operacoes_atu - _operacoes_ant)
        / _operacoes_ant.replace(0, np.nan)
        * 100
    )

    # ── Validação: crescimento de penetração ────────────────────────────────
    # Se o saldo cresce mas o número de operações NÃO cresce (diferença > 20 pp),
    # interpretamos como concentração de risco → penalizamos o crescimento.
    #
    #   crescimento_validado = cresc_saldo × fator_concentracao
    #   fator_concentracao ∈ [0.5, 1.0]
    #     → 1.0 quando as duas taxas crescem na mesma proporção
    #     → 0.5 quando o saldo sobe muito mas operações ficam paradas
    _diff_cresc = _cresc_saldo - _cresc_ops
    _fator_concentracao = np.where(
        _diff_cresc > 20,          # saldo subiu ≥20 pp a mais que operações
        0.5,                       # penalidade de concentração
        1.0,                       # crescimento saudável
    )

    _cresc_validado = (_cresc_saldo * _fator_concentracao).rename("crescimento_validado_pct")

    # Clip de outliers (percentis 5–95) + log-weight pelo tamanho da carteira
    _carteira_total_idx = _agg_total.set_index("estado")["carteira_ativa_total"]
    _cresc_ajustado = (
        _cresc_validado
        * np.log1p(_carteira_total_idx)
    ).pipe(lambda s: s.clip(s.quantile(0.05), s.quantile(0.95)))

    _df_cresc = pd.DataFrame({
        "estado": _cresc_ajustado.index,
        "crescimento_validado_pct": _cresc_validado.values,
        "crescimento_ajustado":     _cresc_ajustado.values,
    })
else:
    print("[AVISO] Menos de 7 períodos disponíveis — crescimento definido como 0.")
    _df_cresc = pd.DataFrame({
        "estado":                  _agg_total["estado"],
        "crescimento_validado_pct": 0.0,
        "crescimento_ajustado":     0.0,
    })

# =============================================================================
# ETAPA 3 — INDICADORES DERIVADOS
# =============================================================================
df_estados = _agg_total.merge(_df_cresc, on="estado", how="left")

# Adiciona população
df_estados["populacao"] = (
    df_estados["estado"].map(POPULACAO_ESTADOS).astype("float32")
)

# ── 3a. Taxa de inadimplência (%): soma/soma → evita média de médias ─────────
df_estados["taxa_inadimplencia_pct"] = (
    df_estados["inadimplencia_total"] / df_estados["carteira_ativa_total"] * 100
).astype("float32")

# ── 3b. Taxa de ativo problemático (%) ───────────────────────────────────────
df_estados["taxa_ativo_problematico_pct"] = (
    df_estados["ativo_problematico_total"] / df_estados["carteira_ativa_total"] * 100
).astype("float32")

# ── 3c. Penetração: operações por habitante ───────────────────────────────────
df_estados["operacoes_por_habitante"] = (
    df_estados["numero_de_operacoes_total"] / df_estados["populacao"]
).fillna(0).astype("float32")

# ── 3d. NOVO — Saldo per capita (endividamento/disponibilidade) ───────────────
# Quanto maior o saldo por habitante, menos espaço de expansão.
# Será normalizado e invertido no score (estados com menor saldo per capita ganham pontos).
df_estados["saldo_per_capita"] = (
    df_estados["carteira_ativa_total"] / df_estados["populacao"]
).fillna(0).astype("float32")

# ── 3e. NOVO — Índice de Estresse de Crédito (IEC) ───────────────────────────
# Combina inadimplência (peso 60 %) e ativo problemático (peso 40 %).
# Representa a saúde financeira do mercado de crédito estadual.
# Valores acima da média nacional receberão penalidade extra no score final.
df_estados["indice_estresse_credito"] = (
    df_estados["taxa_inadimplencia_pct"]      * 0.60
    + df_estados["taxa_ativo_problematico_pct"] * 0.40
).astype("float32")

# =============================================================================
# ETAPA 4 — NORMALIZAÇÃO Min-Max (0–100)
# =============================================================================

def normalizar(serie: pd.Series, inverter: bool = False) -> pd.Series:
    """
    Normaliza para 0–100 via Min-Max.
    inverter=True  → score alto = valor baixo (penaliza indicadores ruins).
    NaNs preenchidos com mediana antes de normalizar.
    """
    s = serie.fillna(serie.median())
    minimo, maximo = s.min(), s.max()
    if maximo == minimo:
        return pd.Series(50.0, index=serie.index, dtype="float32")
    norm = (s - minimo) / (maximo - minimo) * 100
    return (100 - norm).astype("float32") if inverter else norm.astype("float32")


# Componentes do score
score_penetracao   = normalizar(df_estados["operacoes_por_habitante"], inverter=False)
score_crescimento  = normalizar(df_estados["crescimento_ajustado"],    inverter=False)
score_estresse     = normalizar(df_estados["indice_estresse_credito"], inverter=True)
score_endividamento = normalizar(df_estados["saldo_per_capita"],        inverter=True)

# =============================================================================
# ETAPA 5 — SCORE FINAL COM PENALIDADE DE ESTRESSE
# =============================================================================
# Penalidade: estados cujo IEC supera a média nacional perdem pontos extras.
# Isso impede que um estado de risco alto passe pela malha apenas por ter
# inadimplência "relativamente" baixa no grupo.

_media_iec_nacional = df_estados["indice_estresse_credito"].mean()
_penalidade = np.where(
    df_estados["indice_estresse_credito"] > _media_iec_nacional,
    PENALIDADE_ACIMA_MEDIA,
    0.0,
)

df_estados["score_expansao"] = (
    (
        score_penetracao    * PESO_PENETRACAO
        + score_crescimento * PESO_CRESCIMENTO
        + score_estresse    * PESO_ESTRESSE
        + score_endividamento * PESO_ENDIVIDAMENTO
        - _penalidade
    )
    .clip(0, 100)   # garante que a penalidade não leve abaixo de 0
    .round(1)
)

# =============================================================================
# ETAPA 6 — CLASSIFICAÇÃO QUALITATIVA
# =============================================================================
LIMIARES = {
    75: "🟢 Ótimo",    # expansão recomendada
    60: "🟡 Bom",      # expansão com atenção
    45: "🟠 Regular",  # expansão cautelosa
     0: "🔴 Evitar",   # risco alto
}

def classificar(score: float) -> str:
    for limiar, rotulo in LIMIARES.items():
        if score >= limiar:
            return rotulo
    return "🔴 Evitar"

df_estados["classificacao"] = df_estados["score_expansao"].apply(classificar)

# Ordena do melhor para o pior candidato à expansão
df_estados = (
    df_estados
    .sort_values("score_expansao", ascending=False)
    .reset_index(drop=True)
)

# =============================================================================
# ETAPA 7 — RESULTADO FINAL
# =============================================================================
_colunas_exibicao = [
    "estado",
    "classificacao",
    "score_expansao",
    "taxa_inadimplencia_pct",
    "taxa_ativo_problematico_pct",
    "indice_estresse_credito",
    "saldo_per_capita",
    "operacoes_por_habitante",
    "crescimento_validado_pct",
    "crescimento_ajustado",
]

print("=" * 75)
print("RANKING DE EXPANSÃO SUSTENTÁVEL DE CRÉDITO POR ESTADO  (v2)")
print("=" * 75)
print(df_estados[_colunas_exibicao].to_string(index=False))
print()
print(f"Total de estados analisados : {len(df_estados)}")
segmento = CLIENTE_FILTRO if CLIENTE_FILTRO else "Todos (PF + PJ)"
print(f"Segmento                    : {segmento}")
print(f"Penalidade IEC > média BR   : -{PENALIDADE_ACIMA_MEDIA} pts")
print(f"Média nacional IEC          : {_media_iec_nacional:.2f}")
print()
print("Distribuição das classificações:")
print(df_estados["classificacao"].value_counts().to_string())

RANKING DE EXPANSÃO SUSTENTÁVEL DE CRÉDITO POR ESTADO  (v2)
estado classificacao  score_expansao  taxa_inadimplencia_pct  taxa_ativo_problematico_pct  indice_estresse_credito  saldo_per_capita  operacoes_por_habitante  crescimento_validado_pct  crescimento_ajustado
    SP       🟢 Ótimo            83.0                3.215008                     6.932877                 4.702156     521172.375000                57.356255                 11.660974            358.666435
    DF       🟢 Ótimo            82.0                3.057686                     5.680532                 4.106824     629346.375000                55.925449                 19.062853            358.666435
    RS         🟡 Bom            70.9                3.261258                     6.754430                 4.658527     521697.187500                55.318146                  6.631523            194.961692
    SC         🟡 Bom            66.5                2.999237                     5.597204                 4.038424  

In [ ]:
display(dt[dt["uf"] == "SP"].info())

<class 'pandas.core.frame.DataFrame'>
Index: 223334 entries, 255188 to 3366007
Data columns (total 8 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   data_base               223334 non-null  datetime64[ns]
 1   uf                      223334 non-null  object        
 2   cliente                 223334 non-null  object        
 3   porte                   223334 non-null  object        
 4   numero_de_operacoes     178336 non-null  object        
 5   carteira_ativa          223334 non-null  float64       
 6   carteira_inadimplencia  223334 non-null  float64       
 7   ativo_problematico      223334 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 15.3+ MB


None

In [ ]:
set(dt.loc[dt["cliente"] == 'PJ', 'porte'].unique())

{'Grande', 'Micro', 'Médio', 'Pequeno'}

In [ ]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3373856 entries, 0 to 3373855
Data columns (total 8 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   data_base               datetime64[ns]
 1   uf                      object        
 2   cliente                 object        
 3   porte                   object        
 4   numero_de_operacoes     object        
 5   carteira_ativa          float64       
 6   carteira_inadimplencia  float64       
 7   ativo_problematico      float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 205.9+ MB
